In [0]:
%sql
-- Create the dim_customer table
CREATE TABLE IF NOT EXISTS retail_catalog.gold.dim_customer
(
    CustomerID STRING,
    CustomerName STRING,
    Region STRING,
    CustomerSegment STRING,
    SignupDate DATE
);

In [0]:
%sql
-- Create the dim_product table
CREATE TABLE IF NOT EXISTS retail_catalog.gold.dim_product
(
    ProductID STRING,
    ProductName STRING,
    Category STRING,
    Brand STRING,
    UnitPrice DECIMAL(10,2)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.gold.dim_store
(
    StoreID STRING,
    StoreName STRING,
    City STRING,
    Region STRING,
    StoreType STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_catalog.gold.fact_sales
(
    SaleID STRING,
    CustomerID STRING,
    ProductID STRING,
    StoreID STRING,
    SaleDate DATE,
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    Discount DECIMAL(10,2),
    SalesAmount DECIMAL(18,2),
    NetSales DECIMAL(18,2),
    OrderStatus STRING,
    PaymentMethod STRING
);

In [0]:
%sql
-- Incrementally populate dim_customer

MERGE INTO retail_catalog.gold.dim_customer AS tgt

USING (
    SELECT
        CustomerID,
        CustomerName,
        Region,
        CustomerSegment,
        SignupDate
    FROM retail_catalog.silver.customers
) AS src

ON tgt.CustomerID = src.CustomerID
WHEN MATCHED THEN
    UPDATE SET
        tgt.CustomerName = src.CustomerName,
        tgt.Region = src.Region,
        tgt.CustomerSegment = src.CustomerSegment,
        tgt.SignupDate = src.SignupDate

WHEN NOT MATCHED THEN
    INSERT (CustomerID, CustomerName, Region, CustomerSegment, SignupDate)
    VALUES (
        src.CustomerID,
        src.CustomerName,
        src.Region,
        src.CustomerSegment,
        src.SignupDate
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1000,0,0,1000


In [0]:
%sql
-- Incrementally populate dim_product

MERGE INTO retail_catalog.gold.dim_product AS tgt

USING (
    SELECT
        ProductID,
        ProductName,
        Category,
        Brand,
        UnitPrice
    FROM retail_catalog.silver.products
) AS src

ON tgt.ProductID = src.ProductID
WHEN MATCHED THEN
    UPDATE SET
        tgt.ProductName = src.ProductName,
        tgt.Category = src.Category,
        tgt.Brand = src.Brand,
        tgt.UnitPrice = src.UnitPrice

WHEN NOT MATCHED THEN
    INSERT (ProductID, ProductName, Category, Brand, UnitPrice)
    VALUES (
        src.ProductID,
        src.ProductName,
        src.Category,
        src.Brand,
        src.UnitPrice
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
200,0,0,200


In [0]:
%sql
-- Incrementally populate dim_store

MERGE INTO retail_catalog.gold.dim_store AS tgt

USING (
    SELECT
        StoreID,
        StoreName,
        City,
        Region,
        StoreType
    FROM retail_catalog.silver.stores
) AS src

ON tgt.StoreID = src.StoreID
WHEN MATCHED THEN
    UPDATE SET
        tgt.StoreName = src.StoreName,
        tgt.City = src.City,
        tgt.Region = src.Region,
        tgt.StoreType = src.StoreType

WHEN NOT MATCHED THEN
    INSERT (
        StoreID,
        StoreName,
        City,
        Region,
        StoreType
    )
    VALUES (
        src.StoreID,
        src.StoreName,
        src.City,
        src.Region,
        src.StoreType
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
20,0,0,20


In [0]:
%sql
-- Populate fact_sales incrementally

MERGE INTO retail_catalog.gold.fact_sales AS tgt

USING (
    SELECT
        s.SaleID,
        s.CustomerID,
        s.ProductID,
        s.StoreID,
        s.SaleDate,
        s.Quantity,
        s.UnitPrice,
        s.Discount,
        s.Quantity * s.UnitPrice AS SalesAmount,
        s.Quantity * s.UnitPrice * (1 - s.Discount) AS NetSales,
        s.OrderStatus,
        s.PaymentMethod
    FROM retail_catalog.silver.sales s
) AS src

ON tgt.SaleID = src.SaleID

WHEN MATCHED THEN
    UPDATE SET
        tgt.CustomerID = src.CustomerID,
        tgt.ProductID = src.ProductID,
        tgt.StoreID = src.StoreID,
        tgt.SaleDate = src.SaleDate,
        tgt.Quantity = src.Quantity,
        tgt.UnitPrice = src.UnitPrice,
        tgt.Discount = src.Discount,
        tgt.SalesAmount = src.SalesAmount,
        tgt.NetSales = src.NetSales,
        tgt.OrderStatus = src.OrderStatus,
        tgt.PaymentMethod = src.PaymentMethod

WHEN NOT MATCHED THEN
    INSERT (
        SaleID, CustomerID, ProductID, StoreID, SaleDate, Quantity,
        UnitPrice, Discount, SalesAmount, NetSales, OrderStatus, PaymentMethod
    )
    VALUES (
        src.SaleID,
        src.CustomerID,
        src.ProductID,
        src.StoreID,
        src.SaleDate,
        src.Quantity,
        src.UnitPrice,
        src.Discount,
        src.SalesAmount,
        src.NetSales,
        src.OrderStatus,
        src.PaymentMethod
    );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
9901,0,0,9901
